# Node2Vec trên mạng PPI (STRING database)

**Mục tiêu:** Học 30-dim embedding cho mỗi protein trong mạng PPI, lưu vào `proceed_data/protein_node2vec`.

**Input cần có:**
- `D:/raw_data/ppi.txt` — file STRING (dạng `9606.ENSP... 9606.ENSP... score`)
- `D:/CAFA6/proceed_data/uniprot_ensembl_mapping.csv` — output từ `uniprot_mapping.py`
- `D:/CAFA6/proceed_data/valid_protein_ids.csv` — output từ `1_get_valid_ids.py`

**Output:**
- `D:/CAFA6/proceed_data/protein_node2vec` — pickle dict `{UniProt_ID: ndarray(30,)}`

> ⚠️ Chạy tuần tự từ trên xuống dưới. Không bỏ qua cell nào.

In [1]:
# Cài đặt thư viện (chỉ cần chạy lần đầu)
!pip install "node2vec>=0.4.3" networkx pandas numpy

In [2]:
import pickle
import time
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from pathlib import Path

import node2vec, gensim, networkx
print(f"node2vec : {node2vec.__version__}")
print(f"gensim   : {gensim.__version__}")
print(f"networkx : {networkx.__version__}")

node2vec : 0.4.3
gensim   : 4.2.0
networkx : 2.6.3


c:\Users\ngoch\.conda\envs\capstone\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ── Cấu hình đường dẫn ──────────────────────────────────────────────────────
PPI_FILE       = Path(r"D:/raw_data/ppi.txt")
MAPPING_FILE   = Path(r"D:/CAFA6/proceed_data/uniprot_ensembl_mapping.csv")
VALID_IDS_FILE = Path(r"D:/CAFA6/proceed_data/valid_protein_ids.csv")
OUTPUT_FILE    = Path(r"D:/CAFA6/proceed_data/protein_node2vec")

# ── Tham số Node2Vec ────────────────────────────────────────────────────────
DIMENSIONS  = 30    # Phải khớp model input_dim=30 trong train_Struct2GO.py
WALK_LENGTH = 30
NUM_WALKS   = 10
P           = 0.8
Q           = 1.2
WORKERS     = 1     # PHẢI để 1 trên Windows — multi-worker gây TerminatedWorkerError
WINDOW      = 10
MIN_COUNT   = 1
EPOCHS      = 5
MIN_SCORE   = 400

for name, path in [("PPI", PPI_FILE), ("MAPPING", MAPPING_FILE), ("VALID_IDS", VALID_IDS_FILE)]:
    status = "✓" if path.exists() else "✗ KHÔNG TÌM THẤY!"
    print(f"  {status} {name}: {path}")

  ✓ PPI: D:\raw_data\ppi.txt
  ✓ MAPPING: D:\CAFA6\proceed_data\uniprot_ensembl_mapping.csv
  ✓ VALID_IDS: D:\CAFA6\proceed_data\valid_protein_ids.csv


In [5]:
# ── Bước 1: Load bảng ánh xạ ENSP → UniProt ────────────────────────────────
print("Bước 1: Đọc dữ liệu ...")

mapping_df = pd.read_csv(MAPPING_FILE)
ensp2uniprot = dict(zip(mapping_df["Ensembl_Protein"], mapping_df["UniProtKB_AC"]))
print(f"  Có {len(ensp2uniprot):,} cặp ENSP → UniProt")

valid_ids = set(pd.read_csv(VALID_IDS_FILE)["Protein_ID"].tolist())
print(f"  Có {len(valid_ids):,} protein hợp lệ (có file .pdb.gz)")

Bước 1: Đọc dữ liệu ...
  Có 19,111 cặp ENSP → UniProt
  Có 20,550 protein hợp lệ (có file .pdb.gz)


In [6]:
# ── Bước 2: Xây dựng đồ thị PPI ─────────────────────────────────────────────
print("Bước 2: Xây đồ thị PPI ...")
G = nx.Graph()

added = skipped = no_map = low_score = 0

with open(PPI_FILE, "r", encoding="utf-8") as fh:
    header = fh.readline()
    has_score = len(header.strip().split()) >= 3

    for line in fh:
        parts = line.rstrip().split()
        if len(parts) < 2:
            continue

        p1_raw, p2_raw = parts[0], parts[1]

        if has_score and len(parts) >= 3:
            try:
                score = int(parts[2])
            except ValueError:
                score = 0
            if score < MIN_SCORE:
                low_score += 1
                continue

        ensp1 = p1_raw.split(".", 1)[-1] if "." in p1_raw else p1_raw
        ensp2 = p2_raw.split(".", 1)[-1] if "." in p2_raw else p2_raw

        uid1 = ensp2uniprot.get(ensp1)
        uid2 = ensp2uniprot.get(ensp2)

        if not uid1 or not uid2:
            no_map += 1
            continue

        if uid1 in valid_ids and uid2 in valid_ids:
            G.add_edge(uid1, uid2)
            added += 1
        else:
            skipped += 1

print(f"  ✓ Đồ thị PPI: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"  ✗ Không map được: {no_map:,} | Thiếu 3D: {skipped:,} | Score<{MIN_SCORE}: {low_score:,}")

if G.number_of_nodes() == 0:
    raise ValueError("Đồ thị rỗng! Kiểm tra lại MIN_SCORE hoặc các file input.")

Bước 2: Xây đồ thị PPI ...
  ✓ Đồ thị PPI: 17,863 nodes, 810,322 edges
  ✗ Không map được: 66,390 | Thiếu 3D: 155,154 | Score<400: 11,856,460


In [7]:
# ── Bước 3: Chạy Node2Vec ────────────────────────────────────────────────────
# Lưu ý: workers=1 là bắt buộc trên Windows.
# Dùng nhiều worker hơn sẽ gây TerminatedWorkerError do joblib multiprocessing.
# Ước tính thời gian: 17K nodes x 10 walks x 30 steps ≈ 10–30 phút với workers=1.
print(f"Bước 3: Chạy Node2Vec (dim={DIMENSIONS}, workers={WORKERS}) ...")
print(f"  Ước tính 10–30 phút, vui lòng đợi...")
t0 = time.time()

n2v = Node2Vec(
    G,
    dimensions=DIMENSIONS,
    walk_length=WALK_LENGTH,
    num_walks=NUM_WALKS,
    p=P,
    q=Q,
    workers=WORKERS,   # BẮT BUỘC = 1 trên Windows
    quiet=False,
)

model = n2v.fit(
    window=WINDOW,
    min_count=MIN_COUNT,
    epochs=EPOCHS,
    sg=1,
)

elapsed = time.time() - t0
print(f"  ✓ Hoàn thành! Thời gian: {elapsed:.1f}s ({elapsed/60:.1f} phút)")

Bước 3: Chạy Node2Vec (dim=30, workers=1) ...
  Ước tính 10–30 phút, vui lòng đợi...


Generating walks (CPU: 1): 100%|██████████| 10/10 [12:37<00:00, 75.78s/it]


  ✓ Hoàn thành! Thời gian: 3705.8s (61.8 phút)


In [8]:
# ── Bước 4: Thu thập embedding và lưu pickle ─────────────────────────────────
print("Bước 4: Thu thập embedding ...")
protein_node2vec = {}
zero_count = 0

for uid in G.nodes():
    key = str(uid)
    if key in model.wv:
        vec = np.array(model.wv[key], dtype=np.float32)
    else:
        vec = np.zeros(DIMENSIONS, dtype=np.float32)
        zero_count += 1
    protein_node2vec[uid] = vec

print(f"  ✓ Có embedding thực: {len(protein_node2vec) - zero_count:,}")
print(f"  ✗ Zero vector      : {zero_count:,}")

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(protein_node2vec, f)

print(f"\n✅ Đã lưu {len(protein_node2vec):,} embeddings → {OUTPUT_FILE}")

Bước 4: Thu thập embedding ...
  ✓ Có embedding thực: 17,863
  ✗ Zero vector      : 0

✅ Đã lưu 17,863 embeddings → D:\CAFA6\proceed_data\protein_node2vec


In [9]:
# ── Bước 5: Kiểm tra kết quả ─────────────────────────────────────────────────
with open(OUTPUT_FILE, "rb") as f:
    check = pickle.load(f)

sample_id = next(iter(check))
print(f"Tổng số protein có embedding : {len(check):,}")
print(f"Ví dụ ID                     : {sample_id}")
print(f"Shape vector                 : {check[sample_id].shape}")
print(f"Giá trị 5 đầu                : {check[sample_id][:5]}")
print(f"Min / Max                    : {check[sample_id].min():.4f} / {check[sample_id].max():.4f}")

Tổng số protein có embedding : 17,863
Ví dụ ID                     : P84085
Shape vector                 : (30,)
Giá trị 5 đầu                : [-0.11626282 -0.23758115 -0.399856    0.30304238  0.2536643 ]
Min / Max                    : -0.6678 / 1.1479
